# 第 6 章：文本分类微调

前几章造出了一个能「续写」的 GPT。本章让它学会**判断情感**——给定一句影评，输出「正面/负面」。这是把通用语言模型改造成**专用任务模型**的标准范式。

## 核心思路（3 步）

1. **改造输出头**：把最后的 `out_head`（vocab→50257 的续写头）换成 `emb_dim→2` 的分类头
2. **冻结 backbone**：GPT 主体不训练（省算力、防过拟合），只训练「最后一层 + 分类头」
3. **微调训练**：用标注好的情感数据，做**分类交叉熵**（而非语言模型的续写交叉熵）

> 主线用自造的小型中文情感 demo 数据验证流程。真实场景用 SST-2 / IMDb（见 bonus 02）。

## 1. 自造情感分类 demo 数据

用中文正/负面短句各若干条，标签 1=正面、0=负面。规模虽小，足以演示分类微调的完整流程。

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import tiktoken

# 自造中文情感数据（1=正面, 0=负面）
POSITIVE = [
    "这部电影非常精彩 我很喜欢",
    "太好看了 剧情感人至深",
    "画面优美 值得推荐",
    "演技出色 故事动人",
    "完美之作 强烈推荐",
    "音乐动听 视觉震撼",
    "节奏紧凑 引人入胜",
    "结局温暖 回味无穷",
]
NEGATIVE = [
    "太糟糕了 浪费时间",
    "剧情无聊 让人失望",
    "画面粗糙 毫无诚意",
    "演技尴尬 故事混乱",
    "简直烂片 不忍直视",
    "噪音刺耳 看不下去",
    "节奏拖沓 昏昏欲睡",
    "结局糟糕 一无是处",
]
texts = POSITIVE + NEGATIVE
labels = [1] * len(POSITIVE) + [0] * len(NEGATIVE)
print(f"样本数: {len(texts)} (正{len(POSITIVE)} / 负{len(NEGATIVE)})")

## 2. 构建数据集与 DataLoader

In [ ]:
class SentimentDataset(Dataset):
    """情感分类数据集：把每条文本编码成定长 token 序列。"""

    def __init__(self, texts, labels, tokenizer, max_len=32, pad_id=50256):
        self.max_len = max_len
        self.pad_id = pad_id
        self.data = []
        for t, l in zip(texts, labels):
            ids = tokenizer.encode(t)[:max_len]
            self.data.append((ids, l))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        ids, label = self.data[i]
        # 右侧 pad 到定长（分类任务对位置不敏感，pad 放最后）
        ids = ids + [self.pad_id] * (self.max_len - len(ids))
        return torch.tensor(ids), torch.tensor(label)


tok = tiktoken.get_encoding("gpt2")
dataset = SentimentDataset(texts, labels, tok, max_len=32)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

x, y = next(iter(dataloader))
print(f"batch: x {tuple(x.shape)}, y {tuple(y.shape)}, 标签分布 {y.tolist()}")

## 3. 改造 GPT 为分类器

**关键改造**：替换 `out_head`。原来输出 `[batch, seq, 50257]`（每个位置预测下一个词），现在我们只用**最后一个 token 的输出**，经分类头映射成 `[batch, num_classes]`。

> 为什么用最后一个 token？因为因果注意力的最后一个 token「看过」整句话，聚合了全部信息，最适合做整体判断。

In [ ]:
from src.gpt import GPTModel, GPT_CONFIG_124M

# 小配置 demo（完整 124M 太慢，与 ch05 一致）
cfg = dict(GPT_CONFIG_124M)
cfg.update({"emb_dim": 128, "n_layers": 2, "n_heads": 4, "context_length": 32})

torch.manual_seed(123)
model = GPTModel(cfg)

# ★ 核心改造：把续写头换成分类头
num_classes = 2
model.out_head = torch.nn.Linear(cfg["emb_dim"], num_classes)

def classify(model, x):
    """取最后一个 token 的输出做分类。"""
    logits = model(x)          # [b, seq, num_classes]
    return logits[:, -1, :]     # [b, num_classes]

# 验证
x = torch.randint(0, cfg["vocab_size"], (4, 32))
out = classify(model, x)
print(f"分类输出: {tuple(out.shape)} (应为 [4, 2])")

## 4. 冻结 backbone（参数高效微调）

GPT 主体（embedding + 前 n-1 层）**冻结**，只训练：最后一层 Transformer 块 + 最终 LayerNorm + 分类头。这样既省显存、又防小数据过拟合。

> 这与「全量微调」相比，可训练参数大幅减少，但效果通常接近——尤其当数据量小时。

In [ ]:
def freeze_backbone(model):
    """冻结除最后一层块+最终norm+分类头之外的所有参数。"""
    for p in model.parameters():
        p.requires_grad = False
    # 解冻最后一个 Transformer 块
    for p in model.trf_blocks[-1].parameters():
        p.requires_grad = True
    # 解冻最终 LayerNorm
    for p in model.final_norm.parameters():
        p.requires_grad = True
    # 分类头本身可训练
    for p in model.out_head.parameters():
        p.requires_grad = True


freeze_backbone(model)
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"总参数:   {total:,}")
print(f"可训练:   {trainable:,} ({100*trainable/total:.1f}%)")
print(f"冻结:     {total-trainable:,}")

## 5. 微调训练循环

和 ch05 预训练的区别：**loss 从「续写交叉熵」换成「分类交叉熵」**。输入是整句 token，目标是该句的情感标签。

In [ ]:
import torch.nn.functional as F

def calc_loss_batch(input_batch, target_batch, model, device):
    """分类损失：用最后一个 token 的 logits 与标签做交叉熵。"""
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = classify(model, input_batch)   # [b, num_classes]
    loss = F.cross_entropy(logits, target_batch)
    return loss


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=5e-4, weight_decay=0.1
)

model.train()
for epoch in range(15):
    total = 0; n = 0
    for x, y in dataloader:
        optimizer.zero_grad()
        loss = calc_loss_batch(x, y, model, device)
        loss.backward()
        optimizer.step()
        total += loss.item(); n += 1
    if epoch % 3 == 0 or epoch == 14:
        print(f"epoch {epoch:2d}: loss {total/n:.4f}")

## 6. 评估准确率

In [ ]:
def calc_accuracy_loader(data_loader, model, device):
    """计算整个 loader 的分类准确率。"""
    model.eval()
    correct = 0; total = 0
    with torch.no_grad():
        for x, y in data_loader:
            x, y = x.to(device), y.to(device)
            logits = classify(model, x)
            pred = logits.argmax(dim=-1)
            correct += (pred == y).sum().item()
            total += len(y)
    return correct / total


acc = calc_accuracy_loader(dataloader, model, device)
print(f"训练集准确率: {100*acc:.0f}%（随机基线 50%）")
print("\n💡 完整流程：换分类头 → 冻结 backbone → 分类交叉熵微调 → 评估。")
print("   真实场景用 SST-2/IMDb 大数据集，并划分 train/val/test（见 bonus 02）。")